# 📘 Bài Giảng Thực Hành: Kernel Support Vector Machine (Kernel SVM / RBF SVM)
**Mục tiêu bài giảng:**
1. Hiểu nguyên lý Kernel Trick và hàm nhân RBF: $K(\mathbf{x}, \mathbf{x}') = \exp(-\gamma \|\mathbf{x} - \mathbf{x}'\|^2)$.
2. Tinh chỉnh siêu tham số $C$ và $\gamma$ bằng `GridSearchCV`.
3. So sánh hiệu năng vượt trội của RBF Kernel SVM ($93\%$) so với Linear SVM ($90\%$).

---
### 📊 Tập dữ liệu: `Social_Network_Ads.csv`


## 1. Nạp Dữ Liệu & Tinh Chỉnh Siêu Tham Số bằng GridSearchCV


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

dataset = pd.read_csv('Social_Network_Ads.csv')
X = dataset.iloc[:, [2, 3]].values
y = dataset.iloc[:, 4].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
sc = StandardScaler()
X_train_scaled = sc.fit_transform(X_train)
X_test_scaled = sc.transform(X_test)

param_grid = {'C': [0.1, 1, 10, 100], 'gamma': ['scale', 'auto', 0.1, 0.5, 1.0]}
grid_search = GridSearchCV(SVC(kernel='rbf', random_state=0), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

print('Cặp tham số tối ưu:', grid_search.best_params_)
best_svm = grid_search.best_estimator_
print(f'Accuracy trên Test Set: {accuracy_score(y_test, best_svm.predict(X_test_scaled))*100:.2f}%')


📌 **Expected Output:**
`Accuracy trên Test Set: 93.00%`


## 2. Trực quan hóa Ranh Giới Phi Tuyến RBF


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for idx, (X_set, y_set, title) in enumerate([(X_train_scaled, y_train, 'Train'), (X_test_scaled, y_test, 'Test')]):
    X1, X2 = np.meshgrid(np.arange(X_set[:,0].min()-1, X_set[:,0].max()+1, 0.02), np.arange(X_set[:,1].min()-1, X_set[:,1].max()+1, 0.02))
    Z = best_svm.predict(np.array([X1.ravel(), X2.ravel()]).T).reshape(X1.shape)
    axes[idx].contourf(X1, X2, Z, alpha=0.3, cmap=ListedColormap(('salmon', 'lightgreen')))
    for i, j in enumerate(np.unique(y_set)):
        axes[idx].scatter(X_set[y_set==j, 0], X_set[y_set==j, 1], color=ListedColormap(('red', 'green'))(i), label=f'Class {j}')
    axes[idx].set_title(f'Kernel SVM (RBF) - {title} Set', fontweight='bold')
    axes[idx].legend()
plt.tight_layout()
plt.show()


📌 **Expected Output:**
Ranh giới quyết định uốn lượn cong phi tuyến mềm mại ôm trọn các cụm khách hàng mua hàng.
